# Exp19c: Problem-Level Agreement Metrics (10 Students)

Computes Problem F1 and Problem Jaccard to match the methodology 
used in the 3-student V1/V2 study.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)

STUDENT_IDS = [
    '10155', '9948', '14189', '14352', '14362',
    '14363', '14374', '14414', '14474', '14499'
]

HUMAN_A_DIR = Path('dataset/Rater_KC_Tags/Rated_KC_V3')
HUMAN_B_DIR = Path('dataset/Rater_KC_Tags/Rated_KC_V3')
LLM_V3_DIR = Path('results/human_validation/llm_v3_10students')

OUTPUT_DIR = Path('results/human_validation/v3_prompt_eval_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KC_COLUMNS = [
    "If/Else", "NestedIf", "While", "For", "NestedFor",
    "Math+-*/", "Math%", "LogicAndNotOr", "LogicCompareNum", "LogicBoolean",
    "StringFormat", "StringConcat", "StringIndex", "StringLen",
    "StringEqual", "CharEqual", "ArrayIndex", "DefFunction"
]

print(f"Students: {len(STUDENT_IDS)}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def find_latest_file(directory, pattern):
    matches = sorted(directory.glob(pattern))
    return matches[-1] if matches else None

def load_gaps(filepath):
    """Load annotation JSON → {pid_str: set of gap KC strings}."""
    if filepath is None or not Path(filepath).exists():
        return {}
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    result = {}
    for pid, val in data.get('annotations', {}).items():
        if isinstance(val, dict):
            gaps = val.get('gaps', [])
        elif isinstance(val, list):
            gaps = val
        else:
            gaps = []
        result[str(pid)] = set(g.strip() for g in gaps if isinstance(g, str) and g.strip())
    return result

# Load all annotations
all_ha = {}
all_hb = {}
all_llm = {}

for sid in STUDENT_IDS:
    ha_file = find_latest_file(HUMAN_A_DIR, f"kc_annotations_Pranay Ghuge_{sid}_*.json")
    hb_file = find_latest_file(HUMAN_B_DIR, f"kc_annotations_Arundhati Das_{sid}_*.json")
    llm_file = LLM_V3_DIR / f"llm_v3_annotations_{sid}.json"
    
    all_ha[sid] = load_gaps(str(ha_file) if ha_file else None)
    all_hb[sid] = load_gaps(str(hb_file) if hb_file else None)
    all_llm[sid] = load_gaps(str(llm_file))
    
    common = set(all_ha[sid].keys()) & set(all_hb[sid].keys()) & set(all_llm[sid].keys())
    print(f"Student {sid}: {len(common)} common problems")

print(f"\nAll annotations loaded.")

In [ ]:
def problem_f1(set_a, set_b):
    """Compute F1 between two gap sets for a single problem.
    
    Treats set_a as reference, set_b as prediction.
    If both empty → F1 = 1.0 (perfect agreement on no gaps).
    If one empty and other not → F1 = 0.0.
    """
    if len(set_a) == 0 and len(set_b) == 0:
        return 1.0
    if len(set_a) == 0 or len(set_b) == 0:
        return 0.0
    
    tp = len(set_a & set_b)
    fp = len(set_b - set_a)
    fn = len(set_a - set_b)
    
    if tp == 0:
        return 0.0
    
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)


def problem_jaccard(set_a, set_b):
    """Compute Jaccard between two gap sets for a single problem.
    
    Returns None if both sets are empty (undefined).
    """
    if len(set_a) == 0 and len(set_b) == 0:
        return None  # Undefined — exclude from average
    
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    
    if union == 0:
        return None
    
    return intersection / union


def compute_kappa_from_binary(decisions_a, decisions_b):
    """Compute Cohen's kappa from two lists of binary decisions."""
    n = len(decisions_a)
    if n == 0:
        return float('nan')
    
    a = np.array(decisions_a)
    b = np.array(decisions_b)
    
    po = np.mean(a == b)
    p1 = np.mean(a)
    p2 = np.mean(b)
    pe = p1 * p2 + (1 - p1) * (1 - p2)
    
    if pe == 1.0:
        return 1.0
    
    return (po - pe) / (1 - pe)


def compute_gwet_ac1(decisions_a, decisions_b):
    """Compute Gwet's AC1 from two lists of binary decisions."""
    n = len(decisions_a)
    if n == 0:
        return float('nan')
    
    a = np.array(decisions_a)
    b = np.array(decisions_b)
    
    po = np.mean(a == b)
    q1 = np.mean(a)
    q2 = np.mean(b)
    q_bar = (q1 + q2) / 2
    pe = 2 * q_bar * (1 - q_bar)
    
    if pe == 1.0:
        return 1.0
    
    return (po - pe) / (1 - pe)


print("Metric functions defined.")

In [ ]:
# Collect per-problem results for all 3 pairs
rows = []

for sid in STUDENT_IDS:
    common_pids = set(all_ha[sid].keys()) & set(all_hb[sid].keys()) & set(all_llm[sid].keys())
    
    for pid in sorted(common_pids, key=lambda x: int(x)):
        ha_gaps = all_ha[sid][pid]
        hb_gaps = all_hb[sid][pid]
        llm_gaps = all_llm[sid][pid]
        
        # Problem-level F1 (symmetric: average both orderings)
        f1_hh = (problem_f1(ha_gaps, hb_gaps) + problem_f1(hb_gaps, ha_gaps)) / 2
        f1_ha_llm = (problem_f1(ha_gaps, llm_gaps) + problem_f1(llm_gaps, ha_gaps)) / 2
        f1_hb_llm = (problem_f1(hb_gaps, llm_gaps) + problem_f1(llm_gaps, hb_gaps)) / 2
        
        # Problem-level Jaccard
        j_hh = problem_jaccard(ha_gaps, hb_gaps)
        j_ha_llm = problem_jaccard(ha_gaps, llm_gaps)
        j_hb_llm = problem_jaccard(hb_gaps, llm_gaps)
        
        # Binary KC decisions for kappa/AC1
        ha_binary = [1 if kc in ha_gaps else 0 for kc in KC_COLUMNS]
        hb_binary = [1 if kc in hb_gaps else 0 for kc in KC_COLUMNS]
        llm_binary = [1 if kc in llm_gaps else 0 for kc in KC_COLUMNS]
        
        rows.append({
            'StudentID': sid,
            'ProblemID': pid,
            'HA_gaps': sorted(ha_gaps),
            'HB_gaps': sorted(hb_gaps),
            'LLM_gaps': sorted(llm_gaps),
            'F1_HH': f1_hh,
            'F1_HA_LLM': f1_ha_llm,
            'F1_HB_LLM': f1_hb_llm,
            'J_HH': j_hh,
            'J_HA_LLM': j_ha_llm,
            'J_HB_LLM': j_hb_llm,
            'HA_binary': ha_binary,
            'HB_binary': hb_binary,
            'LLM_binary': llm_binary,
        })

df = pd.DataFrame(rows)
print(f"Total problems: {len(df)}")
print(f"Problems with at least one gap: {len(df[df.apply(lambda r: len(r['HA_gaps']) > 0 or len(r['HB_gaps']) > 0 or len(r['LLM_gaps']) > 0, axis=1)])}")

In [ ]:
# --- Problem-Level F1 (average across all problems) ---
avg_f1_hh = df['F1_HH'].mean()
avg_f1_ha_llm = df['F1_HA_LLM'].mean()
avg_f1_hb_llm = df['F1_HB_LLM'].mean()
avg_f1_llm = (avg_f1_ha_llm + avg_f1_hb_llm) / 2

# --- Problem-Level Jaccard (average across problems where at least one rater tagged a gap) ---
j_hh_valid = df['J_HH'].dropna()
j_ha_llm_valid = df['J_HA_LLM'].dropna()
j_hb_llm_valid = df['J_HB_LLM'].dropna()

avg_j_hh = j_hh_valid.mean()
avg_j_ha_llm = j_ha_llm_valid.mean()
avg_j_hb_llm = j_hb_llm_valid.mean()
avg_j_llm = (avg_j_ha_llm + avg_j_hb_llm) / 2

# --- Kappa and AC1 (per-KC binary, pooled across all problems) ---
all_ha_binary = [b for row in df['HA_binary'] for b in row]
all_hb_binary = [b for row in df['HB_binary'] for b in row]
all_llm_binary = [b for row in df['LLM_binary'] for b in row]

kappa_hh = compute_kappa_from_binary(all_ha_binary, all_hb_binary)
kappa_ha_llm = compute_kappa_from_binary(all_ha_binary, all_llm_binary)
kappa_hb_llm = compute_kappa_from_binary(all_hb_binary, all_llm_binary)
kappa_llm = (kappa_ha_llm + kappa_hb_llm) / 2

ac1_hh = compute_gwet_ac1(all_ha_binary, all_hb_binary)
ac1_ha_llm = compute_gwet_ac1(all_ha_binary, all_llm_binary)
ac1_hb_llm = compute_gwet_ac1(all_hb_binary, all_llm_binary)
ac1_llm = (ac1_ha_llm + ac1_hb_llm) / 2

# --- Build summary table ---
summary_rows = [
    {
        'Comparison': 'H-A vs H-B (Ceiling)',
        'Cohen_kappa': round(kappa_hh, 3),
        'Gwet_AC1': round(ac1_hh, 3),
        'Problem_F1': round(avg_f1_hh, 3),
        'Jaccard': round(avg_j_hh, 3),
        'N_Problems': len(df),
        'N_Jaccard_Problems': len(j_hh_valid),
    },
    {
        'Comparison': 'HA vs LLM V3',
        'Cohen_kappa': round(kappa_ha_llm, 3),
        'Gwet_AC1': round(ac1_ha_llm, 3),
        'Problem_F1': round(avg_f1_ha_llm, 3),
        'Jaccard': round(avg_j_ha_llm, 3),
        'N_Problems': len(df),
        'N_Jaccard_Problems': len(j_ha_llm_valid),
    },
    {
        'Comparison': 'HB vs LLM V3',
        'Cohen_kappa': round(kappa_hb_llm, 3),
        'Gwet_AC1': round(ac1_hb_llm, 3),
        'Problem_F1': round(avg_f1_hb_llm, 3),
        'Jaccard': round(avg_j_hb_llm, 3),
        'N_Problems': len(df),
        'N_Jaccard_Problems': len(j_hb_llm_valid),
    },
    {
        'Comparison': 'AvgHuman vs LLM V3',
        'Cohen_kappa': round(kappa_llm, 3),
        'Gwet_AC1': round(ac1_llm, 3),
        'Problem_F1': round(avg_f1_llm, 3),
        'Jaccard': round(avg_j_llm, 3),
        'N_Problems': len(df),
        'N_Jaccard_Problems': int((len(j_ha_llm_valid) + len(j_hb_llm_valid)) / 2),
    },
]

summary_df = pd.DataFrame(summary_rows)

print("=== Agreement Metrics (Problem-Level F1 & Jaccard) ===")
print(summary_df[['Comparison', 'Cohen_kappa', 'Gwet_AC1', 'Problem_F1', 'Jaccard']].to_string(index=False))

In [ ]:
# --- Ceiling analysis ---
ceiling_rows = [
    {
        'Metric': 'Kappa',
        'Human Ceiling': round(kappa_hh, 3),
        'LLM Avg': round(kappa_llm, 3),
        'Gap': round(kappa_hh - kappa_llm, 3),
        '% of Ceiling': f"{100 * kappa_llm / kappa_hh:.1f}%",
    },
    {
        'Metric': 'Problem F1',
        'Human Ceiling': round(avg_f1_hh, 3),
        'LLM Avg': round(avg_f1_llm, 3),
        'Gap': round(avg_f1_hh - avg_f1_llm, 3),
        '% of Ceiling': f"{100 * avg_f1_llm / avg_f1_hh:.1f}%",
    },
    {
        'Metric': 'Jaccard',
        'Human Ceiling': round(avg_j_hh, 3),
        'LLM Avg': round(avg_j_llm, 3),
        'Gap': round(avg_j_hh - avg_j_llm, 3),
        '% of Ceiling': f"{100 * avg_j_llm / avg_j_hh:.1f}%",
    },
    {
        'Metric': 'Gwet AC1',
        'Human Ceiling': round(ac1_hh, 3),
        'LLM Avg': round(ac1_llm, 3),
        'Gap': round(ac1_hh - ac1_llm, 3),
        '% of Ceiling': f"{100 * ac1_llm / ac1_hh:.1f}%",
    },
]

ceiling_df = pd.DataFrame(ceiling_rows)

print("\n=== Ceiling Comparison ===")
print(ceiling_df.to_string(index=False))

# --- Comparison with old V2 results ---
print("\n=== Old V2 Results (3 students, 146 problems) for reference ===")
print("AvgHuman vs V2 Baseline:  κ=0.426  AC1=0.932  Problem_F1=0.811  Jaccard=0.781")
print(f"AvgHuman vs V3 (10 stud): κ={kappa_llm:.3f}  AC1={ac1_llm:.3f}  Problem_F1={avg_f1_llm:.3f}  Jaccard={avg_j_llm:.3f}")

In [ ]:
# Save
summary_df.to_csv(OUTPUT_DIR / 'problem_level_metrics.csv', index=False)

summary_payload = {
    'human_ceiling': summary_rows[0],
    'llm_vs_human_a': summary_rows[1],
    'llm_vs_human_b': summary_rows[2],
    'llm_average': summary_rows[3],
    'ceiling': ceiling_rows,
    'n_students': len(STUDENT_IDS),
    'n_total_problems': len(df),
    'methodology': 'Problem-level F1 and Jaccard (set-based, matching V1/V2 study)',
}

with open(OUTPUT_DIR / 'problem_level_summary.json', 'w') as f:
    json.dump(summary_payload, f, indent=2)

print(f"\nSaved to {OUTPUT_DIR}")
print("  - problem_level_metrics.csv")
print("  - problem_level_summary.json")